In [1]:
import pandas as pd
import uncertainties as un
from uncertainties import unumpy as unp
from scipy.stats import t
import numpy as np

In [2]:
meas = pd.read_hdf("measurements.h5")

In [5]:
sim_and_meas_file = "simulated_and_measured_2024-11-16_gri30_highT.h5"
sim_meas = pd.read_hdf(sim_and_meas_file)
for ((phi, dil_mf, diluent), data) in meas.groupby(["phi_nom", "dil_mf_nom", "diluent"]):
    cell_size = unp.uarray(data["cell_size"], data["u_cell_size"]).mean()
    fixed_uncert = cell_size.std_dev * t.ppf(0.975, len(data)-1)
    sim_meas.loc[
        (sim_meas["phi_nom"] == phi) & (sim_meas["dil_mf_nom"] == dil_mf) & (sim_meas["diluent"] == diluent),
        "u_cell_size_measured",
    ] = fixed_uncert
    
with pd.HDFStore(sim_and_meas_file) as store:
    store.put("data_fixed_uncert", sim_meas)

/tmp/ipykernel_796647/2630746650.py:12: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['diluent', 'znd_step', 'znd_end_time', 'znd_tries',
       'znd_max_temp_time'],
      dtype='object')]

  store.put("data_fixed_uncert", sim_meas)
